Tools
Models can request to call that perform fetching of data from database, searching the web or running code.Tools are pairing of:
1. A schema including the name of the tool a description and argument(often a json schema)
2. A function to excute

In [14]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:llama-3.1-8b-instant")

response=model.invoke("write me an eassay on AI")
response

AIMessage(content='**The Evolution and Impact of Artificial Intelligence**\n\nArtificial Intelligence (AI) has been a rapidly evolving field of research and development, transforming the way we live, work, and interact with one another. From its humble beginnings as a theoretical concept to its current state of widespread implementation, AI has come a long way, and its impact on society is only continuing to grow.\n\n**Early Beginnings**\n\nThe concept of AI dates back to the 1950s, when computer scientists and mathematicians first began exploring the possibility of creating machines that could think and learn like humans. The term "Artificial Intelligence" was coined by John McCarthy in 1956, and the field of study was formalized with the creation of the Dartmouth Summer Research Project on Artificial Intelligence. The early years of AI research focused on developing algorithms and techniques for machine learning, natural language processing, and computer vision.\n\n**Advances in AI T

In [15]:
## Tools

from langchain.tools import tool
@tool
def get_weather(location:str)->str:
    """Get the weather at a location """
    return f"it's sunny at in {location}"

model_with_tools=model.bind_tools([get_weather])

In [16]:
response=model_with_tools.invoke("whats the weather in banaglore")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': '1nse3jcz6', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.024597225, 'completion_tokens_details': None, 'prompt_time': 0.014147263, 'prompt_tokens_details': None, 'queue_time': 0.049522676, 'total_time': 0.038744488}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f9a69-b9fc-77f2-abf9-15e996dc178b-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': '1nse3jcz6', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235}


In [23]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Bangalore'},
  'id': '1nse3jcz6',
  'type': 'tool_call'}]

In [32]:
from typing import final
#step1:MODEL GENERATES TOOL CALLS
messages = [{"role":"user", "content":"whats the weather in bangalore?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#step2: execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #execute the total with the generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)
#step 3: pass the result back to model for final response
final_response=model_with_tools.invoke(messages)
print(final_response.text)

I can't verify the current weather in Bangalore.


In [33]:
model_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E51FBF00D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E51FBCFA90>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather 